# Troubleshooting and FAQ

Start with the symptom. Each entry says what it means and how to fix it; most have a cell that reproduces the problem
so you can see the fix work.

| Symptom | Section |
| --- | --- |
| `immune.verdict(...)` returns `None` | [1](#1.-immune.verdict(...)-returns-None) |
| Every verdict says `sensor: tier0_only`, or warnings say `sensor unavailable` | [2](#2.-Jev-isn't-being-used) |
| Immune changed a reply it shouldn't have | [3](#3.-A-legitimate-reply-was-changed) |
| `immune.ConfigError` at startup | [4](#4.-ConfigError-at-startup) |
| A new site appears for every request | [5](#5.-A-new-site-for-every-request) |
| Workers behave differently | [6](#6.-Workers-disagree) |
| `immune.Blocked` is raised | [7](#7.-immune.Blocked-is-raised) |
| Tests try to reach the network | [8](#8.-Tests-reach-the-network) |

`immune doctor` checks your installation, configuration, keys and vaccines in one go; `--live` adds a real Jev call
(in a terminal, drop the `!`):

In [1]:
!immune doctor --live

check             ok   detail
────────────────  ───  ─────────────────────────────────────────────────
python            yes  3.14.4
spec              yes  2026.10.1 (49 threats)
TYPESAFE_API_KEY  yes  set
typesafe-sdk      yes  0.7.2
openai            yes  3.22.1
anthropic         yes  1.9.0
google-genai      yes  not installed
httpx2            yes  2.13.1
httpx             yes  0.28.1
litellm           yes  not installed
langchain-core    yes  1.6.6
interception      yes  httpx2 (every transport), httpx (every transport)
vaccines          yes  0 loaded
state             yes  /tmp/guide-prerun-s8wyb1hy/immune-home
jev               yes  jev-1.13.0 in 449 ms


## 1. `immune.verdict(...)` returns `None`

The call wasn't screened, or the verdict was looked up the wrong way.

| Cause | Fix |
| --- | --- |
| `immune.init()` was never called, or `IMMUNE_DISABLED=1` is set | Call it once at startup; check the environment |
| `mode: off`, or the site has `enabled: false` | Expected: nothing is screened |
| The call goes to a gateway on a path Immune doesn't recognize | Add an `endpoints` route (below) |
| LangChain or LangGraph: `immune.verdict()` read inside the `with immune.site(...)` block | Read it after the block, or pass the message: `immune.verdict(message)` |
| Concurrent code: `immune.verdict()` without an argument | Pass the response: `immune.verdict(response)` |

**A gateway on a custom path.** The OpenAI SDK always calls `…/chat/completions` under its `base_url`, which Immune
recognizes on any host. If your code posts to a gateway on its own path (for example `POST /llm/complete` with an
OpenAI-style body), tell Immune which wire format it speaks:

```yaml
endpoints:
  - openai_chat=^/llm/complete$
```

Formats: `openai_chat`, `openai_responses`, `anthropic_messages`, `gemini`.

## 2. Jev isn't being used

`sensor: tier0_only` means Jev wasn't asked; the log says why (`immune: sensor unavailable for …`).

| Log message | Cause | Fix |
| --- | --- | --- |
| `… no API key` / `401` | `TYPESAFE_API_KEY` missing or wrong | Set it in the environment the app runs in |
| `sensor exceeded 0.80s` | Jev was slower than `sensor.timeout_ms` for that request | Occasional misses are expected; raise `timeout_ms` if they are frequent |
| `sensor circuit breaker is open` | Many recent failures | Fix the cause; the breaker retries after `breaker_cooldown_s` |
| `… shed` | Your Jev quota was exhausted | Add keys to `sensor.api_keys` or raise `requests_per_minute` |

While Jev is unavailable, floor candidates still act (`sensor.on_outage: act_on_candidates`), and each such hit's
evidence says "acted without Jev". A verdict can also show `0` Jev requests with `sensor: jev` when identical requests
were answered from Immune's short-lived cache; that is normal.

## 3. A legitimate reply was changed

1. Read `verdict.enforced_hits`: which threat acted, with what evidence.
2. Report it: `immune.feedback(verdict.trace_id, "false_positive", threat=...)`. Labels tune the detector for your
   traffic (`immune calibrate`, `immune evidence fit`).
3. While you investigate, observe that threat at that site. This takes effect on the next call:

In [2]:
import immune

immune.init(config={"sites": {"support-chat": {"enforce": ["input.off_task"]}}})
site = immune.runtime().settings.sites["support-chat"]
print(f"before: enforce={list(site.enforce)} observe={list(site.observe)}")

immune.configure(sites={"support-chat": {"enforce": [], "observe": ["input.off_task"]}})
site = immune.runtime().settings.sites["support-chat"]
print(f"after:  enforce={list(site.enforce)} observe={list(site.observe)}")

before: enforce=['input.off_task'] observe=[]
after:  enforce=[] observe=['input.off_task']


Floor rules can be observed per site too; Immune logs a warning, and `immune doctor` reports it, so it isn't
forgotten. If the passage is something customers should see (a menu, a policy), keep it in a clearly labelled section
of the system prompt, or in your knowledge base.

## 4. `ConfigError` at startup

| Message | Fix |
| --- | --- |
| `Extra inputs are not permitted` | A misspelled key; the message names it (`sites.support-chat.enforced`) |
| `Input should be 'auto', 'observe', 'strict' or 'off'` | A wrong value; the message lists the allowed ones |
| `would disable floor protections` | Narrow the `vaccines.disabled` pattern, or set `vaccines.allow_floor_changes: true` |
| `configuration file … does not exist` | Check `IMMUNE_CONFIG` or the path passed to `immune.init(config=...)` |
| `… cannot change at runtime; call immune.init() again` | `state`, `sensor`, `state_dir`, `canary` and `vaccines.paths` need a restart |
| `choose exactly one detector` (vaccines) | A vaccine's `detect` needs exactly one of keywords/regex, questions, tool, python |

## 5. A new site for every request

Immune names unnamed sites by fingerprinting the system prompt and tools. If your system prompt includes per-request
text (the customer's name, today's date, retrieved documents), each request looks new. Name the site:

In [3]:
import tempfile
from pathlib import Path

import immune
from immune.testing import FakeReply, ImmuneHarness

harness = ImmuneHarness(Path(tempfile.mkdtemp()), script=FakeReply("Hi!"))
for customer in ["Ana", "Jordan"]:
    messages = [{"role": "system", "content": f"You are helping {customer}, a Pinecrest customer since 2021."}]
    unnamed = harness.openai().chat.completions.create(
        model="m", messages=[*messages, {"role": "user", "content": "Hi"}]
    )
    with immune.site("support-chat"):
        named = harness.openai().chat.completions.create(
            model="m", messages=[*messages, {"role": "user", "content": "Hi"}]
        )
    print(f"{customer:<7} unnamed site: {harness.verdict(unnamed).site:<14} named site: {harness.verdict(named).site}")
harness.close()

Ana     unnamed site: site-24d9838a  named site: support-chat
Jordan  unnamed site: site-65d067bd  named site: support-chat


## 6. Workers disagree

Each process keeps its own state unless you share it: a confirmation requested on one worker won't be found on
another, and promotions happen per process. Set `state: {backend: redis, url: ...}` (or `sqlite` on one host), or
`IMMUNE_STATE_URL`. See [Production](15_production.ipynb#1.-Several-workers-share-one-state).

## 7. `immune.Blocked` is raised

`on_block: raise` is set, so blocked calls raise instead of returning a safe reply. The exception carries the
verdict, and for OpenAI clients it is also an `openai.OpenAIError`:

```python
try:
    response = client.chat.completions.create(...)
except immune.Blocked as error:
    log.info("blocked: %s", error.verdict.explanation)
    reply = "Sorry, I can't help with that."
```

## 8. Tests reach the network

Use `ImmuneHarness` (or the `immune_harness` pytest fixture), which runs a scripted model and a scripted Jev
offline. To turn Immune off entirely for unrelated test suites, set `IMMUNE_DISABLED=1`. See
[Testing and CI](13_testing_and_ci.ipynb).

## Common mistakes

| Avoid | Prefer instead |
| --- | --- |
| `immune.init()` per request | Once at startup |
| Retrieved documents in the system prompt | User message with `immune.untrusted(...)`, or tool results |
| `immune.verdict()` in concurrent code | `immune.verdict(response)` |
| Enforcing every detector on day one | `auto`, read `would_action`, enforce per site |
| A new session id per call | One per conversation or thread |
| Disabling a floor rule for one noisy case | `allowed_destinations`, `observe` at one site, or feedback |
| Keys in `immune.yaml` | `TYPESAFE_API_KEY` in the environment |

## FAQ

**Does Immune change my prompts?**
Only in three cases: it strips hidden or smuggled text from user and data content when Jev confirms it is aimed at the
model (F1); it adds a short reference line to your system prompt (`canary: true`, optional); and it replaces a data
item with a notice when it carries instructions with very high confidence (F7).

**How much latency does it add?**
Input screening runs in parallel with your model call. Output screening adds one Jev round trip after the model
finishes, about 300–450 ms at the median, bounded by `sensor.timeout_ms` (800 ms). Streams release text as it
arrives until a candidate appears.

**What does it cost?**
Jev tokens: about 2,500 per screened call, plus about 600–700 when a call has candidates. There is no charge from the
Immune project.

**Does it store my conversations?**
Its state holds per-session security data (whether the session read outside content, destinations and where they
came from, pending confirmations, risk) and site profiles. For OpenAI Responses `previous_response_id` chains it keeps
the chain's recent turns, masked, for up to `limits.session_ttl_s` (a day), so later calls can be screened with their
history. The verdict log is off unless you set `privacy.verdict_log`.

**What leaves my process?**
To TypeSafe, for screening: the latest user message and recent turns, new data items, the reply and tool calls, with
personal data and secrets masked first; and once per new site, a masked template of your system prompt
(`privacy.profiling: local` avoids that). Nothing is sent to the Immune project.

**Which providers and frameworks does it cover?**
OpenAI (Chat Completions, Responses, Azure), Anthropic, Google Gemini and Vertex, Amazon Bedrock Converse, and
OpenAI-compatible servers; and frameworks built on those SDKs: LangChain, LangGraph, LlamaIndex, LiteLLM, DSPy, the
OpenAI Agents SDK, Pydantic AI, Instructor and others. `immune.coverage()` lists what is intercepted in your process.

**Can I use it without Jev?**
Yes, but only floor candidates act, without Jev's judgment, and the Jev-based detectors don't run. Immune is designed
around Jev; plan on a key in production.

**Is prompt injection solved?**
No, and Immune never claims it is. It reduces risk in layers (data checks, output checks, tool-call checks, session
memory) so that a single miss doesn't become an incident.

**Which languages does it support?**
Screening questions are tuned for English. The reflexes that find candidates work on any text.

**Is it safe with threads and asyncio?**
Yes. Sessions and sites use context variables, verdicts are tracked per call, and callbacks run on a background
thread.

**Can it break my app?**
It is designed not to: internal errors fail open, Jev outages fall back to floor candidates, and replacement
responses are valid provider responses of the usual SDK types.

Still stuck? Run `immune doctor --live`, and include its output and the verdict's `trace_id`, `spec_version` and
`config_hash` when you open an issue.